In [14]:
from functools import reduce
from pathlib import Path
import sys
_project_root: Path = Path.cwd().resolve()
sys.path.insert(0, str(_project_root.parents[1]))
from jax import config
config.update("jax_enable_x64", True)
import jax
import jax.numpy as jnp
from jax.scipy.linalg import expm

from src.core import Array
import src.symmetry as sym
import src.quantum_symmetry as qsym


In [ ]:
Q8 = qsym.quaternion_group_rep()
I2 = Q8.matrices["1"]
X = -1j * Q8.matrices["k"]
Y = -1j * Q8.matrices["j"]
Z = -1j * Q8.matrices["i"]

In [ ]:
# ---- utils

def _H(A: Array) -> Array:
  return jnp.conjugate(A.T)


def vec(A: Array) -> Array:
  """
  Column-major vectorization using row-major primitives.

  vec([[a,b],
        [c,d]]) = [a,c,b,d]
  """
  return A.T.reshape((-1,))


def unvec(v: Array, d: int) -> Array:
  """ Inverse of column-major vectorization. """
  return v.reshape((d, d)).T

def kron_all(ops: list[Array]) -> Array:
  return reduce(jnp.kron, ops)

def local_op(op: Array, site: int, n: int) -> Array:
  ops = [I2 for _ in range(n)]
  ops[site] = op
  return kron_all(ops)

def collective(op: Array, n: int, coeff: float = 1.0) -> Array:
  at_site = lambda i: local_op(op, i, n)
  return coeff * jnp.array(sum(map(at_site, range(n))))



def comm(A: Array, B: Array) -> Array:
  return A @ B - B @ A


def anticomm(A: Array, B: Array) -> Array:
  return A @ B + B @ A


def outer(psi: Array, phi: Array | None = None) -> Array:
  if phi is None:
    phi = psi
  return psi[:, None] @ jnp.conjugate(phi[None, :])


def Sx(n: int) -> Array:
  return 0.5 * collective(X, n)


def Sy(n: int) -> Array:
  return 0.5 * collective(Y, n)


def Sz(n: int) -> Array:
  return 0.5 * collective(Z, n)

def S2_tot(n: int) -> Array:
  sx = Sx(n)
  sy = Sy(n)
  sz = Sz(n)
  return sx @ sx + sy @ sy + sz @ sz


def nullspace_basis(A: Array, tol: float = 1e-8) -> Array:
  U, S, Vh = jnp.linalg.svd(A)
  rank = int(jnp.sum(S > tol))
  # Nullspace basis lives in rows of Vh after rank
  return _H(Vh)[..., rank:]


def four_qubit_singlet_basis(tol: float = 1e-8) -> Array:
  J2 = S2_tot(4)
  return nullspace_basis(J2, tol=tol)

We start in the initial state 
$$
|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt 2}.
$$
The initial density matrix is 
$$
\rho(0)=\frac12\begin{pmatrix} 1 & 1 \\ 1& 1\end{pmatrix}
$$
Under spontaneous emission, we have that $\rho_{11}(t)=\frac12 e^{-\gamma t}$ and $|\rho_{01}(t)|=\frac12 e^{-\gamma t/2}$

In [ ]:




def lindblad_rhs(H: Array, Ls: list[Array], rho: Array) -> Array:
  """
  Dense Lindblad RHS:

      L(rho) = -i[H,rho]
                + sum_a L_a rho L_a^†
                - 1/2 {L_a^† L_a, rho}
  """
  # ---- -i [H, ρ]
  L_0 = -1j * comm(H, rho)

  def add_term(L: Array, L_k: Array) -> Array:
    LdL = _H(L_k) @ L_k
    return L + L_k @ rho @ _H(L_k) - 0.5 * anticomm(LdL, rho)

  return reduce(add_term, Ls, initial=L_0)


def lindblad_superoperator(H: Array, Ls: list[Array]) -> Array:
  """
  Dense Liouville-space Lindblad matrix using column-major vec.

  Convention:

      vec(A rho B) = (B^T ⊗ A) vec(rho)
  """
  d = H.shape[0]
  I = jnp.eye(d, dtype=H.dtype)

  # Hamiltonian part: -i [H, ρ]
  Lsuper = -1j * (jnp.kron(I, H) - jnp.kron(H.T, I))

  def add_term(L: Array, L_k: Array):
    LdL = _H(L_k) @ L_k
    # vec(L rho L†) = ((L†)^T ⊗ L) vec(rho) = (L* ⊗ L) vec(rho)
    jump = jnp.kron(jnp.conjugate(L_k), L_k)
    # vec(L†L rho) = (I ⊗ L†L) vec(rho)
    left_decay = jnp.kron(I, LdL)
    # vec(rho L†L) = ((L†L)^T ⊗ I) vec(rho)
    right_decay = jnp.kron(LdL.T, I)

    return L + jump - 0.5 * (left_decay + right_decay)

  return reduce(add_term, Ls, Lsuper)

def evolve_liouville(Lsuper: Array, ρ_0: Array, t: float) -> Array:
  d = ρ_0.shape[0]
  rho_t_vec = expm(t * Lsuper) @ vec(ρ_0)
  return unvec(rho_t_vec, d)

In [ ]:
# --- running and testing
def trace_error(rho: Array) -> float:
  return jnp.abs(jnp.trace(rho) - 1.0).item()

def hermiticity_error(rho: Array) -> float:
  return jnp.linalg.norm(rho - _H(rho))


def main():
  dtype = jnp.complex128
  # TODO container for 0, 1 and+, - bases
  # Basis states: |0>, |1>
  ψ_0 = jnp.array([1.0, 0.0], dtype=dtype)
  ψ_1 = jnp.array([0.0, 1.0], dtype=dtype)

  # Lowering operator σ_- = |0><1|
  σ_m = outer(ψ_0, ψ_1)

  γ: float = 1.0

  # Interaction-picture spontaneous emission: H = 0
  H = jnp.zeros((2, 2), dtype=dtype)
  L = jnp.sqrt(γ) * σ_m
  Ls = [L]

  # Initial state: |+> = (|0> + |1>)/sqrt(2)
  ψ_p = (ψ_0 + ψ_1) / jnp.sqrt(2.0)
  ρ_0 = outer(ψ_p)

  # Direct RHS
  rho_dot_direct = lindblad_rhs(H, Ls, ρ_0)

  # Liouville superoperator
  Lsuper = lindblad_superoperator(H, Ls)
  rho_dot_liouville = unvec(Lsuper @ vec(ρ_0), d=2)

  rhs_error = jnp.linalg.norm(rho_dot_direct - rho_dot_liouville)

  print("Initial rho:")
  print(ρ_0)
  print()

  print("Direct Lindblad RHS:")
  print(rho_dot_direct)
  print()

  print("Liouville RHS:")
  print(rho_dot_liouville)
  print()

  print("RHS consistency error:")
  print(rhs_error)
  print()

  # Evolve at a few times
  ts = jnp.array([0.0, 0.5, 1.0, 2.0, 4.0, 8.0])

  print("Time evolution:")
  print("t        rho_00        rho_11        |rho_01|      trace_err    herm_err")

  for t in ts:
      rho_t = evolve_liouville(Lsuper, ρ_0, float(t))

      p0 = jnp.real(rho_t[0, 0])
      p1 = jnp.real(rho_t[1, 1])
      coherence = jnp.abs(rho_t[0, 1])

      print(
          f"{float(t):.2f}    "
          f"{float(p0):.6f}    "
          f"{float(p1):.6f}    "
          f"{float(coherence):.6f}    "
          f"{float(trace_error(rho_t)):.2e}    "
          f"{float(hermiticity_error(rho_t)):.2e}"
      )


if __name__ == "__main__":
  main()

[[0.+0.j 1.+0.j]
 [0.+0.j 0.+0.j]]
Initial rho:
[[0.5+0.j 0.5+0.j]
 [0.5+0.j 0.5+0.j]]

Direct Lindblad RHS:
[[ 0.5 +0.j -0.25+0.j]
 [-0.25+0.j -0.5 +0.j]]

Liouville RHS:
[[ 0.5 +0.j -0.25+0.j]
 [-0.25+0.j -0.5 +0.j]]

RHS consistency error:
0.0

Time evolution:
t        rho_00        rho_11        |rho_01|      trace_err    herm_err
0.00    0.500000    0.500000    0.500000    2.22e-16    0.00e+00
0.50    0.696735    0.303265    0.389400    2.22e-16    0.00e+00
1.00    0.816060    0.183940    0.303265    2.22e-16    0.00e+00
2.00    0.932332    0.067668    0.183940    2.22e-16    0.00e+00
4.00    0.990842    0.009158    0.067668    2.22e-16    0.00e+00
8.00    0.999832    0.000168    0.009158    2.22e-16    0.00e+00


# Collective Dephasing

An environment does not look at each qubit separately, but only couples to the total $z$-spin of the two-qubit system. The noise operator is either 
$$
L = Z_1 +Z_2
$$
or,  with the physics normalization,
$$
S_z = \frac12 (Z_1+Z_2)
$$

# Cyclic Ring Demo

In [ ]:
def cyclic_shift_unitary(n: int, direction: str = "right") -> Array:
    d = 2**n
    U = jnp.zeros((d, d), dtype=jnp.complex128)

    for x in range(d):
        bits = [(x >> (n - 1 - i)) & 1 for i in range(n)]

        if direction == "right":
            shifted = [bits[-1]] + bits[:-1]
        elif direction == "left":
            shifted = bits[1:] + [bits[0]]
        else:
            raise ValueError("direction must be 'left' or 'right'")

        y = 0
        for b in shifted:
            y = (y << 1) | b

        U = U.at[y, x].set(1.0)

    return U

# ---- a cyclic-symmetric Hamiltonian
def heisenberg_ring_H(n: int, J: float = 1.0) -> Array:
    H = 0
    for i in range(n):
        j = (i + 1) % n
        H = H + J * (
            local_two_site(X, i, X, j, n)
            + local_two_site(Y, i, Y, j, n)
            + local_two_site(Z, i, Z, j, n)
        )
    return H


def local_two_site(A: Array, i: int, B: Array, j: int, n: int) -> Array:
    ops = [I2 for _ in range(n)]
    ops[i] = A
    ops[j] = B
    return kron_all(ops)

16. The main conceptual warning

The finite group symmetry decomposition acts on Liouville space:

B(H)

not merely on

H.

So the sectors you get from R
g
	​

=U
g
∗
	​

⊗U
g
	​

 are sectors of operators/density matrices. They tell you how density-matrix components evolve. That is valuable for block diagonalizing the Lindbladian.